# Lab 55 (solution): Calibrated detection and judgment

Reference implementation. Two stand-ins from earlier labs, both replaced by calibration on held-out data: the fixed cosine threshold of [Lab 50](../../50-closing-the-failure-loop/) (Part A) and the single additive judge shift + all-dimensions gate of [Lab 51](../../51-calibrated-multidimensional/) (Part B). The math - ROC / Youden's $J$ and isotonic regression / PAVA - is in [math-foundations/15](../../../math-foundations/15-calibration-threshold-selection.md).

## Step 0: Setup

In [ ]:
from calibrate import (embed, cosine, make_change_pairs, tune_threshold,
                       qwk, additive_shift, isotonic_fit, isotonic_predict, weighted_gate)
# Two stand-ins replaced by calibration on held-out data: a fixed cosine threshold (Lab 50) and
# a single additive judge shift + all-dims gate (Lab 51). Math: math-foundations/15.
print("Part A: tune the change-detection threshold | Part B: isotonic judge calibration + weighted gate")

## Part A: tune the change-detection threshold (item 2)

In [ ]:
# Part A (item 2): real embeddings put reflows and meaning-changes on OVERLAPPING cosine ranges,
# so a fixed 0.98 cutoff cries wolf on reflows. Tune it on labeled pairs by maximizing Youden's J.
pairs = make_change_pairs()
cos_labels = [(cosine(embed(a), embed(b)), lbl) for a, b, lbl in pairs]
sames = [c for c, label in cos_labels if label == 0]
changed = [c for c, label in cos_labels if label == 1]
print(f"reflow cosine  {min(sames):.3f}-{max(sames):.3f}   changed cosine {min(changed):.3f}-{max(changed):.3f}  (overlap)")
res = tune_threshold(cos_labels)
f, t = res["fixed_0_98"], res["tuned"]
print(f"fixed 0.98 : acc {f['acc']:.2f}  FPR {f['fpr']:.2f}  (flags reflows as changes)")
print(f"tuned {res['threshold']:.3f}: acc {t['acc']:.2f}  FPR {t['fpr']:.2f}  (J-maximizing cutoff)")

## Part B: isotonic judge calibration (item 3)

In [ ]:
# Part B (item 3): the judge's bias is monotone but NOT a constant offset - it compresses the
# low end (gold 0,1,2,3 -> judge 0,0,1,2). An additive shift cannot bend; isotonic (PAVA) can.
N = 24
gold = [(i * 7) % 4 for i in range(N)]
compress = {0: 0, 1: 0, 2: 1, 3: 2}
judge = [compress[g] for g in gold]
calib = list(range(12))
test = list(range(12, 24))
shift=additive_shift([judge[i] for i in calib],[gold[i] for i in calib])
add_cal=[max(0,min(3,judge[i]+shift)) for i in range(N)]
xs,ys=isotonic_fit([judge[i] for i in calib],[gold[i] for i in calib])
iso_cal=[isotonic_predict(xs,ys,judge[i]) for i in range(N)]
print("isotonic map judge->gold:", [(x,round(y,2)) for x,y in zip(xs, ys, strict=False)])
raw=qwk([judge[i] for i in test],[gold[i] for i in test])
add=qwk([add_cal[i] for i in test],[gold[i] for i in test])
iso=qwk([iso_cal[i] for i in test],[gold[i] for i in test])
print(f"completeness QWK on TEST:  raw {raw:.2f}  ->  additive {add:.2f}  ->  isotonic {iso:.2f}")
print("the additive shift overshoots the compressed low end; isotonic fits the shape.")

## Part B: a weighted multi-dimensional gate

In [ ]:
# A weighted multi-dimensional gate replaces all-dimensions-pass. Weights say which dimensions
# the product owner values; one strong dimension can offset a weaker one. This is a product
# decision, not a statistical one.
F = [3, 3, 2, 1]
R = [3, 2, 3, 1]
C = [1, 2, 1, 3]
sbd = {"f": F, "r": R, "c": C}
W = {"f": 0.5, "r": 0.3, "c": 0.2}
def all_dims(i):
    return F[i] >= 2 and R[i] >= 2 and C[i] >= 2
def wgate(i):
    return weighted_gate(sbd, W, i, threshold=0.66)
for i in range(4):
    print(f"  release {i}: F{F[i]} R{R[i]} C{C[i]}  all-dims={'PASS' if all_dims(i) else 'BLOCK':5s}  weighted={'PASS' if wgate(i) else 'BLOCK'}")
print("\nRelease 0 (strong F/R, weak C) passes the weighted gate but fails all-dims - the gate now")
print("reflects that faithfulness matters more than completeness here.")

## What you built

Two stand-ins replaced by calibration on held-out data. **Part A**: the change detector's cosine threshold is tuned on labeled reflow/edit pairs by maximizing Youden's $J$, instead of a guessed 0.98. With real (overlapping) embedding distributions the fixed cutoff has a 0.33 false-positive rate - it flags a third of reflows as changes - while the tuned cutoff cuts that to 0.08 at higher accuracy. **Part B**: the judge's bias is monotone but not a constant offset, so an additive shift can't fix it; isotonic regression (pool-adjacent-violators) fits a monotone map judge -> gold and recovers more quadratic-weighted agreement (0.70 raw -> 0.88 additive -> 0.92 isotonic). And the all-dimensions-pass gate becomes a weighted gate, so a release strong on faithfulness can offset a weaker dimension - a product decision the weights make explicit.

**Where this simplifies:** the embedder is a deterministic char-trigram stand-in so the lab runs offline - production passes a sentence-transformer, and the *threshold-tuning procedure* is identical (label pairs, sweep, maximize $J$). PAVA is implemented inline to show the algorithm; `sklearn.isotonic.IsotonicRegression` is the production fit. The calibration and threshold are fit on a held-out split here too - refit them when the embedder or judge model changes, the same way you would retrain any model. The math is in [math-foundations/15](../../../math-foundations/15-calibration-threshold-selection.md).